# Unit Test Generation — Autoresearch
PhD Research: Plain LLM vs Simple RAG vs Iterative Critique RAG for unit test generation.

**Steps:**
1. Install dependencies
2. Install & start Ollama, pull model
3. Clone repo
4. One-time setup (download dataset + build knowledge base)
5. Run experiment
6. View results

## Step 1: Install Python dependencies

In [ ]:
!pip install -q ollama datasets sentence-transformers scikit-learn nltk rouge beautifulsoup4 numpy pandas

## Step 2: Install Ollama and pull model

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in background
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)  # wait for server to be ready
print('Ollama server started.')

In [ ]:
# Choose your model based on available GPU RAM:
# T4  (15GB): llama3.2:latest, phi3:mini, qwen2.5:3b, deepseek-coder:6.7b
# A100(40GB): llama3.1:8b, phi4:14b, qwen2.5:14b

MODEL = 'llama3.2:latest'   # change this if needed
!ollama pull {MODEL}

## Step 3: Clone the repo

In [ ]:
import os

REPO_DIR = '/content/autoresearch'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/balajivenky06/autoresearch.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())
!ls

## Step 4: One-time setup (download dataset + build knowledge base)

In [ ]:
# Downloads HumanEval + MBPP from HuggingFace (~2 min)
# Fetches pytest/unittest docs and encodes them (~2 min)
# Only needs to run once per Colab session
!python prepare_unitest.py

## Step 5: Configure and run a single experiment
Use `set_experiment()` to pick METHOD, REASONING, and model. Then run the next cell.

In [ ]:
import re, ast

VALID_METHODS    = {'plain_llm', 'simple_rag', 'iterative_critique'}
VALID_REASONINGS = {'base', 'cot', 'tot', 'got'}

def set_experiment(method='plain_llm', reasoning='base', model=MODEL):
    """Update train_unitest.py config in-place."""
    assert method in VALID_METHODS, f'method must be one of {VALID_METHODS}'
    assert reasoning in VALID_REASONINGS, f'reasoning must be one of {VALID_REASONINGS}'

    with open('train_unitest.py', 'r') as f:
        content = f.read()

    content = re.sub(r'^METHOD\s*=.*$',         'METHOD    = "' + method    + '"', content, flags=re.MULTILINE)
    content = re.sub(r'^REASONING\s*=.*$',       'REASONING = "' + reasoning + '"', content, flags=re.MULTILINE)
    content = re.sub(r'^GENERATOR_MODEL\s*=.*$', 'GENERATOR_MODEL = "' + model + '"', content, flags=re.MULTILINE)
    content = re.sub(r'^HELPER_MODEL\s*=.*$',    'HELPER_MODEL    = "' + model + '"', content, flags=re.MULTILINE)

    # Validate result is still valid Python before writing
    try:
        ast.parse(content)
    except SyntaxError as e:
        raise RuntimeError('set_experiment produced invalid Python: ' + str(e))

    with open('train_unitest.py', 'w') as f:
        f.write(content)
    print('Config set: method={}, reasoning={}, model={}'.format(method, reasoning, model))

# Set your experiment here:
set_experiment(method='plain_llm', reasoning='base', model=MODEL)

In [ ]:
# Run the experiment
!python train_unitest.py 2>&1 | tee run_unitest.log

## Step 6: View results

In [ ]:
!grep -E '^val_score:|^method:|^model:|^avg_' run_unitest.log

## Step 7: Run all 12 experiments (full PhD comparison)
3 methods x 4 reasoning techniques. **~2 hours on T4.**

In [ ]:
import subprocess, os, pandas as pd

EXPERIMENTS = [
    ('plain_llm',          'base'),
    ('plain_llm',          'cot'),
    ('plain_llm',          'tot'),
    ('plain_llm',          'got'),
    ('simple_rag',         'base'),
    ('simple_rag',         'cot'),
    ('simple_rag',         'tot'),
    ('simple_rag',         'got'),
    ('iterative_critique', 'base'),
    ('iterative_critique', 'cot'),
    ('iterative_critique', 'tot'),
    ('iterative_critique', 'got'),
]

results = []
os.makedirs('/content/experiment_logs', exist_ok=True)
sep = '=' * 50

for method, reasoning in EXPERIMENTS:
    print('\n' + sep)
    print('Running: {} / {}'.format(method, reasoning))
    print(sep)

    set_experiment(method=method, reasoning=reasoning, model=MODEL)

    log_path = '/content/experiment_logs/{}_{}.log'.format(method, reasoning)
    try:
        ret = subprocess.run(
            ['python', 'train_unitest.py'],
            capture_output=True, text=True, timeout=900
        )
        with open(log_path, 'w') as f:
            f.write(ret.stdout + ret.stderr)
        stdout = ret.stdout
    except subprocess.TimeoutExpired:
        print('  TIMEOUT — skipping')
        results.append({'method': method, 'reasoning': reasoning, 'val_score': 0.0, 'status': 'timeout'})
        continue

    val_score = 0.0
    for line in stdout.splitlines():
        if line.startswith('val_score:'):
            try:
                val_score = float(line.split()[1])
            except Exception:
                pass

    status = 'ok' if val_score > 0 else 'crash'
    results.append({'method': method, 'reasoning': reasoning, 'val_score': val_score, 'status': status})
    print('  val_score: {:.6f}  [{}]'.format(val_score, status))

# Summary
df = pd.DataFrame(results).sort_values('val_score', ascending=False)
print('\n=== FINAL RESULTS ===')
print(df.to_string(index=False))
df.to_csv('/content/experiment_logs/summary.csv', index=False)
print('\nSaved to /content/experiment_logs/summary.csv')

## Step 8: Save results to Google Drive (optional)

In [ ]:
from google.colab import drive
drive.mount('/drive')

import shutil
dest = '/drive/MyDrive/PhD_Results/unit_test_autoresearch'
shutil.copytree('/content/experiment_logs', dest, dirs_exist_ok=True)
print('Saved to', dest)